# Notebook 5: Advanced SBI Methods

In this notebook, we will use a 2-step process to first find an optimal summary network through an embedding network using VMIM and then train a second estimator on the likelihood of the compressed data.

## Concept

For many applications in large scale structure cosmology, the dimensionality of the data vector becomes to large, necessitating the use of compression methods. However, many cosmological simulations don't follow the prior one would want to choose, making NPE unfeasible. A method to get around this is too train NPE with an embedding network and then learn the likelihood of the data compressed by the emedding network.

We will also use a more flexible approach for training and MCMC sampling, allowing for greater customization.

## Methods

We will explore a technique called VMIM using the methods implemented in the `sbi` package:
1.  **NPE of raw data**: Trains an embedding network using the VMIM loss.
2.  **NLE**: Learn likelihood of compressed ata and then use MCMC to sample from the posterior.

## Task: Log-Normal Toy Model

We will infer parameters in a log-normal toy model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional
from copy import deepcopy
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sbi.neural_nets.net_builders import build_nsf, build_mdn
from sbi.inference import likelihood_estimator_based_potential
from sbi.inference.posteriors import DirectPosterior
from sbi.utils import BoxUniform

import emcee
from chainconsumer import ChainConsumer, Chain, Truth, PlotConfig

## Create Simulations

In [ ]:
# 1. define simulator

def model_fn(theta):
    theta_pow = (theta**np.arange(1,4)[None,:])
    theta_mult = np.ones((theta.shape[0], 3, 10)) * np.arange(1, 31).reshape(3, 10)[np.newaxis, ...]
    return (theta_pow[:,:,np.newaxis] * theta_mult).reshape(-1, 30)

cov = np.eye(30)
for i in range(29):
    cov[i, i+1] = 0.02
    cov[i+1, i] = 0.02

def generate_data(n_samples=1000, theta=None, grid='wide'):

    if theta is None:

        if grid == 'wide':
            theta = np.random.uniform(-2, 2, (n_samples, 3))
        elif grid == 'narrow':
            theta = np.random.uniform(-1, 1, (n_samples, 3))
        elif grid == 'combined':
            theta_narrow = np.random.uniform(-1, 1, (int(n_samples//2), 3))
            theta_wide = np.random.uniform(-2, 2, (int(n_samples//2), 3))
            theta = np.concatenate([theta_narrow, theta_wide], axis=0)
        else:
            raise ValueError(f"Unsupported grid: {grid}")

    x = model_fn(theta)
    noise = np.exp(np.random.multivariate_normal(np.ones(30), cov, n_samples)) - 4.464546442601885
    return torch.tensor(x + noise).float(), torch.tensor(theta).float()


In [ ]:
# Training data doesn't follow Prior
x_train_npe, theta_train_npe = generate_data(10000, grid='combined')
x_val_npe, theta_val_npe = generate_data(1000, grid='combined')
x_train_nle, theta_train_nle = generate_data(5000, grid='combined')
x_val_nle, theta_val_nle = generate_data(1000, grid='combined')

# Test data follows Prior
x_test, theta_test = generate_data(1000, grid='wide')

# We create a separate set at a fiducial model: theta = [0,0,0]
x_fiducial, theta_fiducial = generate_data(100, np.zeros((100,3)))

In [ ]:
prior = BoxUniform(-2*torch.ones(3), 2*torch.ones(3))

## NPE with embedding network

In [ ]:
class FCNet(nn.Module):

    def __init__(
        self,
        input_dim: int = 1440,
        hidden_features: int = 1024,
        output_dim: int = 18,
        num_blocks: int = 2,
        dropout: float = 0.1,
        norm_type: Optional[str] = 'layer',
        activation: type[nn.Module] = nn.ReLU
    ) -> None:

        super().__init__()

        self.block_list = nn.ModuleList()

        if norm_type == 'layer':
            self.norm = nn.LayerNorm(input_dim)
        elif norm_type == 'batch':
            self.norm = nn.BatchNorm1d(input_dim)
        elif norm_type is not None:
            raise ValueError(f"Unsupported norm_type: {norm_type}")

        for _ in range(num_blocks):
            in_features = input_dim if _ == 0 else hidden_features
            block = nn.Sequential(
                nn.Linear(in_features, hidden_features),
                activation(inplace=True),
                nn.Dropout(dropout),
            )
            self.block_list.append(block)

        self.head = nn.Linear(hidden_features, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "norm"):
            x = self.norm(x)
        for block in self.block_list:
            x = block(x)
        return self.head(x)

In [ ]:
embedding_net = FCNet(input_dim=30, hidden_features=32, num_blocks=2,  output_dim=6, dropout=0.1)

density_estimator_npe = build_mdn(  theta_train_npe,
                                    x_train_npe,
                                    z_score_x='independent',
                                    z_score_y='independent',
                                    hidden_features=64,
                                    num_blocks=2,
                                    num_components=1,
                                    dropout_probability=0.,
                                    embedding_net=embedding_net,
                                    device='cpu')

dataset_training_npe = TensorDataset(theta_train_npe, x_train_npe)
dataset_validation_npe = TensorDataset(theta_val_npe, x_val_npe)

train_loader_kwargs = {
    "batch_size": 50,
    "drop_last": True,
    "shuffle": True,
}

val_loader_kwargs = {
    "batch_size": 50,
    "shuffle": False,
    "drop_last": True,
}

train_loader = DataLoader(dataset_training_npe, **train_loader_kwargs)
val_loader = DataLoader(dataset_validation_npe, **val_loader_kwargs)

### Now the Training Loop

In [ ]:
optimizer = Adam(list(density_estimator_npe.parameters()), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

epoch, val_loss = 0, float("Inf")
best_model_state_dict = {}
epochs_since_last_improvement = 0
best_val_loss = float("Inf")
converged = False
max_num_epochs = 1000

while epoch <= max_num_epochs and not converged:

    print(f"Training NLE epoch {epoch}...", end='\r')

    density_estimator_npe.train()
    train_loss_sum = 0
    for batch in train_loader:
        optimizer.zero_grad()
        x_batch, theta_batch = (
            batch[0],
            batch[1],
        )

        train_losses = density_estimator_npe.loss(x_batch, condition=theta_batch)
        train_loss = torch.mean(train_losses)
        train_loss_sum += train_losses.sum().item()

        train_loss.backward()

        optimizer.step()

    epoch += 1

    # Calculate validation performance.
    density_estimator_npe.eval()
    val_loss_sum = 0

    with torch.no_grad():
        for batch in val_loader:
            x_batch, theta_batch = (batch[0], batch[1])

            val_losses = density_estimator_npe.loss(
                x_batch,
                condition=theta_batch,
            )
            val_loss_sum += val_losses.sum().item()

    scheduler.step(val_loss_sum)

    # Take mean over all validation samples.
    val_loss = val_loss_sum / (len(val_loader) * val_loader.batch_size)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_since_last_improvement = 0
        best_model_state_dict = deepcopy(density_estimator_npe.state_dict())
    else:
        epochs_since_last_improvement += 1

    # If no validation improvement over many epochs, stop training.
    if epochs_since_last_improvement > 25:
        density_estimator_npe.load_state_dict(best_model_state_dict)
        converged = True
        print(f'NPE successfully converged after {epoch} epochs')
        print(f'Best validation loss: {best_val_loss:.4f}')

# Make sure model is in eval mode after training is finished
_ = density_estimator_npe.eval()

### Now use the trained embedding network to compress the NLE training Data

In [ ]:
with torch.inference_mode():
    x_train_nle_compressed = embedding_net(x_train_nle)
    x_val_nle_compressed = embedding_net(x_val_nle)
    x_test_compressed = embedding_net(x_test)
    x_fiducial_compressed = embedding_net(x_fiducial)

## Now Train NLE

Here we first define the density estimator using a Neural Spline Flow (NSF) and then train it using a custom training loop.

In [ ]:
# Create NSF (you can also change the build function to build_maf for comparison)
density_estimator_nle = build_nsf(  x_train_nle_compressed,
                                    theta_train_nle,
                                    z_score_x='independent',
                                    z_score_y='independent',
                                    hidden_features=64,
                                    num_blocks=3,
                                    num_transforms=8,
                                    dropout_probability=0.,
                                    num_bins=6,
                                    hidden_layers=2,
                                    device='cpu')

dataset_training_nle = TensorDataset(x_train_nle_compressed, theta_train_nle)
dataset_validation_nle = TensorDataset(x_val_nle_compressed, theta_val_nle)

train_loader_kwargs = {
    "batch_size": 50,
    "drop_last": True,
    "shuffle": True,
}

val_loader_kwargs = {
    "batch_size": 50,
    "shuffle": False,
    "drop_last": True,
}

train_loader = DataLoader(dataset_training_nle, **train_loader_kwargs)
val_loader = DataLoader(dataset_validation_nle, **val_loader_kwargs)

In [ ]:
optimizer = Adam(list(density_estimator_nle.parameters()), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

epoch, val_loss = 0, float("Inf")
best_model_state_dict = {}
epochs_since_last_improvement = 0
best_val_loss = float("Inf")
converged = False
max_num_epochs = 1000

while epoch <= max_num_epochs and not converged:

    print(f"Training NLE epoch {epoch}...", end='\r')

    density_estimator_nle.train()
    train_loss_sum = 0
    for batch in train_loader:
        optimizer.zero_grad()
        x_batch, theta_batch = (
            batch[0],
            batch[1],
        )

        train_losses = density_estimator_nle.loss(x_batch, condition=theta_batch)
        train_loss = torch.mean(train_losses)
        train_loss_sum += train_losses.sum().item()

        train_loss.backward()

        optimizer.step()

    epoch += 1

    # Calculate validation performance.
    density_estimator_nle.eval()
    val_loss_sum = 0

    with torch.no_grad():
        for batch in val_loader:
            x_batch, theta_batch = (batch[0], batch[1])

            val_losses = density_estimator_nle.loss(
                x_batch,
                condition=theta_batch,
            )
            val_loss_sum += val_losses.sum().item()

    scheduler.step(val_loss_sum)

    # Take mean over all validation samples.
    val_loss = val_loss_sum / (len(val_loader) * val_loader.batch_size)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_since_last_improvement = 0
        best_model_state_dict = deepcopy(density_estimator_nle.state_dict())
    else:
        epochs_since_last_improvement += 1

    # If no validation improvement over many epochs, stop training.
    if epochs_since_last_improvement > 25:
        density_estimator_nle.load_state_dict(best_model_state_dict)
        converged = True
        print(f'NLE successfully converged after {epoch} epochs')
        print(f'Best validation loss: {best_val_loss:.4f}')

if not converged:
    print(f"Reached maximum number of epochs: {max_num_epochs}")
    density_estimator_nle.load_state_dict(best_model_state_dict)

# Make sure model is in eval mode after training is finished
_ = density_estimator_nle.eval()

## Create posterior sample using emcee

In [ ]:
potential_fn, _ = likelihood_estimator_based_potential(density_estimator_nle, prior, x_o=x_fiducial_compressed.mean(axis=0))

nwalkers = 1000
nsteps = 1000
discard = 10000
thin = 3
tot_steps = nsteps * thin + discard

def log_prob(args):
    return np.array(potential_fn(torch.tensor(args).float()).tolist())

theta_0 = prior.sample([nwalkers])

sampler = emcee.EnsembleSampler(nwalkers, 3, log_prob, vectorize=True)
state = sampler.run_mcmc(theta_0, tot_steps, progress=True)
samples = sampler.get_chain(discard=discard, thin=thin, flat=True)

# Also sample from NPE for comparison
post_NPE = DirectPosterior(density_estimator_npe, prior)
samples_npe = post_NPE.sample((1000000,), x_fiducial.mean(axis=0)).detach().numpy()

In [ ]:

c = ChainConsumer()
param_names = [rf"$\theta_{i+1}$" for i in range(3)]

post_df = pd.DataFrame(samples, columns=param_names)
post_npe_df = pd.DataFrame(samples_npe, columns=param_names)

truth_dict = dict(zip(param_names, theta_fiducial[0]))


c.add_chain(Chain(samples=post_df, color='red', name="NLE", sigmas=[0,1,2]))
c.add_chain(Chain(samples=post_npe_df, color='blue', name="NPE", sigmas=[0,1,2]))
c.add_truth(Truth(location=truth_dict))

c.set_plot_config(
    PlotConfig(
        contour_label_font_size=36,
        label_font_size=18,
        tick_font_size=16,
    )
)

fig = c.plotter.plot()


## Now test Coverage using tarp

You will be able to see that NLE with the learned summary statistics is well-calibrated even though the training data did not follow the prior. On the other hand, NPE is not well-calibrated since the density estimator implicitly assumes that the data follows the prior.

*Note: This can take a long time to run (multiple hours)*

In [ ]:
from tarp import get_tarp_coverage

# 1. Define Test Set
# By default all test sims
# num_test = 200 might be faster for testing
num_test = theta_test.shape[0]

# 2. Sample from Posterior for each x_test
# We need a batch of posterior samples for each x in x_test.
# NPE allows batch sampling directly.

posterior_samples_NLE = []
posterior_samples_NPE = []

# Sampling loop (can be slow for many test points)
with tqdm(total=num_test) as pbar:
    for i in range(num_test):

        # Create Likelihood Function for MCMC sampling for each (test) observation
        def log_prob(args):
            return np.array(potential_fn(torch.tensor(args).float()).tolist())

        # MCMC sampling
        theta_0 = prior.sample([100])
        sampler = emcee.EnsembleSampler(100, 3, log_prob, vectorize=True)
        state = sampler.run_mcmc(theta_0, 6000, progress=False)
        posterior_samples_NLE.append(sampler.get_chain(discard=500, thin=2, flat=True))

        # Also sample from NPE for comparison
        posterior_samples_NPE = post_NPE.sample((5000,), x_test[i].unsqueeze(0)).detach().numpy()

        pbar.update(1)

# Stack: [num_test, num_samples, num_params]
posterior_samples_NLE = torch.stack(posterior_samples_NLE)
posterior_samples_NPE = torch.stack(posterior_samples_NPE)

# TARP expects samples to be [num_samples, num_sims, num_dims]
# Currently posterior_samples is [num_test (sims), num_samples, num_params]
# So we transpose to [num_samples, num_test, num_params]
posterior_samples_NLE = posterior_samples_NLE.transpose(0, 1)
posterior_samples_NPE = posterior_samples_NPE.transpose(0, 1)

# 3. Compute Coverage using TARP
# theta_test: [num_test, num_params]
# posterior_samples: [num_samples, num_test, num_params]
ecp_NLE, alpha_NLE = get_tarp_coverage(posterior_samples_NLE.numpy(), theta_test.numpy(), references='random', metric='euclidean', num_alpha_bins=20, num_bootstrap=100, bootstrap=True)
ecp_NPE, alpha_NPE = get_tarp_coverage(posterior_samples_NPE.numpy(), theta_test.numpy(), references='random', metric='euclidean', num_alpha_bins=20, num_bootstrap=100, bootstrap=True)

# 4. Plot P-P Plot
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
plt.errorbar(alpha_NLE, ecp_NLE.mean(axis=0), yerr=ecp_NLE.std(axis=0), label='NLE')
plt.errorbar(alpha_NPE, ecp_NPE.mean(axis=0), yerr=ecp_NPE.std(axis=0), label='NPE')
plt.xlabel('Credibility Level')
plt.ylabel('Expected Coverage')
plt.title('TARP Coverage Test')
plt.legend()
plt.show()